# Convex hull lab

A convex hull wraps a set of points with the smallest convex boundary. In this lab, you'll see which points become hull vertices, how interior and collinear points are handled, and why numerical precision matters.

This notebook uses the repository's Graham scan teaching implementation. Its orientation predicate uses floating-point arithmetic, so it is not a robust exact geometry kernel.

In [ ]:
from pathlib import Path
import sys
package_paths = [
    Path.cwd() / "implementations/python/convex-hull",
    Path.cwd().parent / "implementations/python/convex-hull",
]
package_path = next((path for path in package_paths if path.is_dir()), None)
if package_path is None:
    raise RuntimeError("Run this notebook from the repository or notebooks directory.")
sys.path.insert(0, str(package_path))

import matplotlib.pyplot as plt
from graham_scan import convex_hull

## 1. See the hull around a point cloud

The scan returns only extreme vertices, counter-clockwise, without repeating the first point. The plotting code closes the line visually.

In [ ]:
points = [
    (0.2, 1.0), (0.8, 4.0), (1.5, 2.0), (2.0, 0.4),
    (2.5, 3.2), (3.0, 1.4), (3.6, 4.2), (4.2, 2.0),
    (4.8, 0.8), (5.0, 3.4), (2.4, 2.1), (3.1, 2.4),
    (1.0, 2.6), (4.0, 3.0), (2.0, 0.4),
]
hull = convex_hull(points)
closed_hull = hull + [hull[0]] if len(hull) > 1 else hull

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter([p[0] for p in points], [p[1] for p in points],
           label="Input points", s=45)
if closed_hull:
    ax.plot([p[0] for p in closed_hull], [p[1] for p in closed_hull],
            marker="o", linewidth=2.5, label="Convex hull")
ax.set_title("Graham scan: extreme points form the boundary")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

print("Unique input points:", len(set(points)))
print("Hull vertices:", len(hull))
print("Counter-clockwise hull:", hull)

## 2. Understand the orientation test

For points O, A and B:

$$
\operatorname{cross}(O,A,B)
=(A_x-O_x)(B_y-O_y)-(A_y-O_y)(B_x-O_x)
$$

A positive value means a counter-clockwise turn; a negative value means clockwise; zero means collinear. Graham scan removes clockwise turns as it builds the boundary.

In [ ]:
def cross(o, a, b):
    return ((a[0] - o[0]) * (b[1] - o[1])
            - (a[1] - o[1]) * (b[0] - o[0]))

examples = [
    ((0, 0), (1, 0), (1, 1)),
    ((0, 0), (1, 1), (1, 0)),
    ((0, 0), (1, 1), (2, 2)),
]
for triple in examples:
    print(triple, "cross =", cross(*triple))

## 3. What happens to collinear points?

This implementation keeps only extreme vertices. Points on a straight hull edge that lie between its endpoints are omitted. Try changing the set and rerunning the cell.

In [ ]:
collinear_edge_points = [
    (0, 0), (1, 0), (2, 0), (2, 1), (2, 2),
    (1, 2), (0, 2), (0, 1), (1, 1),
]
print("Hull:", convex_hull(collinear_edge_points))

## 4. Degenerate and duplicate inputs

A point set may contain duplicates, one point, two points or an entirely collinear sequence. These are not afterthoughts: the output contract should state what happens in each case.

In [ ]:
cases = {
    "empty": [],
    "one point": [(2, 3)],
    "two points": [(2, 3), (0, 1)],
    "all collinear": [(0, 0), (1, 1), (2, 2), (3, 3)],
    "duplicates": [(0, 0), (1, 0), (0, 1), (0, 0), (1, 0)],
}
for label, sample in cases.items():
    print(label, "->", convex_hull(sample))

## 5. Complexity and numerical caution

Polar sorting takes $O(n\log n)$ time. The scan processes each sorted point a constant number of times, so the full method is $O(n\log n)$.

The implementation uses floating-point orientation tests. When three points are nearly collinear, rounding may change the sign. Robust geometry libraries use exact or adaptive predicates for high-assurance work. A teaching example should make that limitation visible rather than hiding it.

## Your turn

1. Add a point strictly inside the hull. Does the hull change?
2. Add a point outside one edge. Which hull vertices are replaced?
3. Add several points on one hull edge. Which ones remain?
4. Make all points collinear. Why are only two returned?
5. Scale every coordinate by 1,000. Does the combinatorial result change?
6. Create almost-collinear triples. Why might ordinary floating-point predicates become unreliable?

**Implementation:** [Graham scan package](../implementations/python/convex-hull/README.md) and [intern walkthrough](../implementations/python/convex-hull/implementation-walkthrough.md).